# Preprocessing for Power BI (Part 2)

In [1]:
import pandas as pd

In [ ]:
# Read from pickle
sorted_patterns = pd.read_pickle('checkpoint.pkl')

## Convert prices

In [3]:
# Conversion rates as of ~April 1, 2026
exchange_rates = {
    'USD': 1.0,

    'HRK': 1/7.04,       # 0.1420 USD
    'GBP': 1/0.75,       # 1.3333 USD
    'EUR': 1/0.86,       # 1.1628 USD
    'AUD': 1/1.45,       # 0.6897 USD
    'CAD': 1/1.39,       # 0.7194 USD
    'NOK': 1/9.72,       # 0.1029 USD
    'DKK': 1/6.45,       # 0.1550 USD
    'NZD': 1/1.74,       # 0.5747 USD
    'HKD': 1/7.84,       # 0.1276 USD
    'JPY': 1/158.77,     # 0.00630 USD
    'SEK': 1/9.42,       # 0.1062 USD
    'CHF': 1/0.79,       # 1.2658 USD
    'HUF': 1/330.59,     # 0.00302 USD
    'ZAR': 1/16.86,      # 0.0593 USD
    'ISK': 1/124.50,     # 0.00803 USD
    'PLN': 1/3.70,       # 0.2703 USD
    'CZK': 1/21.16,      # 0.0473 USD
    'RUB': 1/80.30,      # 0.01245 USD
    'BRL': 1/5.16,       # 0.1938 USD
    'SGD': 1/1.28,       # 0.7813 USD
    'ILS': 1/3.13,       # 0.3195 USD
    'MXN': 1/17.84,      # 0.0561 USD
    'INR': 1/93.21,      # 0.01073 USD
    'TWD': 1/32.01       # 0.03124 USD
}

In [4]:
# Convert prices to USD
sorted_patterns['price_usd'] = sorted_patterns.apply(
    lambda row: row['price'] * exchange_rates.get(row['currency'], 1.0) if pd.notna(row['price']) and pd.notna(row['currency']) else None,
    axis=1
)

In [5]:
sorted_patterns = sorted_patterns.drop(columns=['price', 'currency'], errors='ignore')

## Calculate mean time between pattern uploads per designer

In [ ]:
# Add column: days_since_previous_pattern = days since publication / days_since_previous_pattern (only including patterns by the same designer)
for designer in sorted_patterns['author_id'].unique():
	designer_patterns = sorted_patterns[sorted_patterns['author_id'] == designer].sort_values('created_at', ascending=True).reset_index()

	previous_pattern_publish_days = None

	for idx, row in designer_patterns.iterrows():
		if idx == 0:
			previous_pattern_publish_days = row['days_since_publication']
		else:
			sorted_patterns.loc[sorted_patterns['pattern_id'] == row['pattern_id'],
					   'days_since_previous_pattern'] = (previous_pattern_publish_days - row['days_since_publication'])

In [7]:
sorted_patterns[['days_since_previous_pattern', 'author_id', 'days_since_publication']].head(10)

,days_since_previous_pattern,author_id,days_since_publication
0,557.0,1,6433
1,935.0,1,6055
2,1909.0,1,5081
3,1909.0,1,5081
4,1909.0,1,5081
5,1909.0,1,5081
6,1909.0,1,5081
7,1909.0,1,5081
8,1909.0,1,5081
9,1909.0,1,5081


## Add estimated revenue per pattern

In [8]:
sorted_patterns['estimated_revenue'] = sorted_patterns['price_usd'] * sorted_patterns['projects_count']

## Get only count of languages

In [9]:
# Total languages the pattern is listed in - sum of all language_ columns
sorted_patterns['total_languages'] = sorted_patterns[[col for col in sorted_patterns.columns if col.startswith('languages_')]].sum(axis=1)

In [10]:
sorted_patterns = sorted_patterns.drop(columns=[col for col in sorted_patterns.columns if col.startswith('languages_')], errors='ignore')

## Convert yarn_fiber_ columns to bool

In [11]:
# For all columns beginning with 'yarn_fiber_', convert to bool
yarn_fiber_cols = [col for col in sorted_patterns.columns if col.startswith('yarn_fiber_')]
sorted_patterns[yarn_fiber_cols] = sorted_patterns[yarn_fiber_cols].fillna(False).astype(bool)

## Remove faulty data

In [ ]:
# Remove rows where features are negative or 0 that don't make sense
sorted_patterns = sorted_patterns[
	(sorted_patterns['projects_count'] >= 0) &
	(sorted_patterns['favorites_count'] >= 0) &
	(sorted_patterns['queued_projects_count'] >= 0) &
	(sorted_patterns['price_usd'] > 0) &
	(sorted_patterns['days_since_previous_pattern'] >= 0)
]

In [13]:
# Remove patterns where yarn_weight == 'Aran / Worsted' or 'DK / Sport' since they do not match expected categories and are very few in number
sorted_patterns = sorted_patterns[~sorted_patterns['yarn_weight'].isin(['Aran / Worsted', 'DK / Sport'])]

In [14]:
# Remove rows where supercategory is not in the top 6 most common supercategories
top_supercategories = sorted_patterns['supercategory'].value_counts().nlargest(6).index
sorted_patterns = sorted_patterns[sorted_patterns['supercategory'].isin(top_supercategories)]

In [15]:
# Fill NaN values in yardage, price_usd, and days_since_previous_pattern with 0
sorted_patterns['yardage'] = sorted_patterns['yardage'].fillna(0)
sorted_patterns['price_usd'] = sorted_patterns['price_usd'].fillna(0)
sorted_patterns['days_since_previous_pattern'] = sorted_patterns['days_since_previous_pattern'].fillna(0)

In [ ]:
# Keep only records where yardage <= 1000000
# Records above this value are very few in number and have been inaccurately entered by the pattern uploader
sorted_patterns = sorted_patterns[sorted_patterns['yardage'] <= 1000000]

# Keep only records where price_usd <= 100
sorted_patterns = sorted_patterns[sorted_patterns['price_usd'] <= 100]

In [17]:
# Convert days_since_previous_pattern to numeric, coercing errors to NaN, then fill NaN with 0
sorted_patterns['days_since_previous_pattern'] = pd.to_numeric(sorted_patterns['days_since_previous_pattern'], errors='coerce').fillna(0)

In [18]:
sorted_patterns['days_since_previous_pattern'].describe()

count    559308.000000
mean       1440.806184
std        1437.530544
min           0.000000
25%         306.000000
50%         986.000000
75%        2165.000000
max        6910.000000
Name: days_since_previous_pattern, dtype: float64

In [19]:
# Convert animal_fiber, vegetable_fiber, and synthetic_fiber to bool
fiber_cols = ['animal_fiber', 'vegetable_fiber', 'synthetic_fiber']
sorted_patterns[fiber_cols] = sorted_patterns[fiber_cols].fillna(False).astype(bool)

# Convert free_patterns to bool
sorted_patterns['free_patterns'] = sorted_patterns['free_patterns'].fillna(False).astype(bool)

## Save Data

In [ ]:
# Save to pickle for future use
sorted_patterns.to_pickle('preprocessed_patterns.pkl')